# PEFT+AFSP — scoring the stacked rungs

---
## 1 — Preconditions

In [134]:
%cd /home/prnamhr/projects/Style-Aware-MT
import json
import os
import subprocess
import sys
from pathlib import Path

import yaml

PY = sys.executable
COMET_PY = '.venv-comet/bin/python'
SPLIT = 'val'
OUT = Path('outputs')

REFERENCE, CONTROL, ARM = 'peft', 'peft_knn', 'peft_afsp'
CONDS = [REFERENCE, CONTROL, ARM]

METRICS = ('chrf', 'bleu', 'comet')

N_BOOT, N_STYLO, ALPHA, SEED = 10000, 2000, 0.05, 42
print(f'{len(CONDS)} conditions, {len(METRICS)} adequacy metrics, {N_BOOT} resamples, seed {SEED}')

/home/prnamhr/projects/Style-Aware-MT
3 conditions, 3 adequacy metrics, 10000 resamples, seed 42


In [135]:
%cd /home/prnamhr/projects/Style-Aware-MT

/home/prnamhr/projects/Style-Aware-MT


In [136]:
VAL = [json.loads(x) for x in Path(f'data/splits/{SPLIT}.jsonl').open(encoding='utf-8') if x.strip()]
SRC = [r['input'] for r in VAL]

ROWS = {}
for cond in CONDS:
    path = OUT / f'{cond}_{SPLIT}.jsonl'
    assert path.exists(), f'{path} missing -- sync it back from the generation box'
    ROWS[cond] = [json.loads(x) for x in path.open(encoding='utf-8') if x.strip()]
    assert len(ROWS[cond]) == len(VAL), f'{cond}: {len(ROWS[cond])} rows, expected {len(VAL)}'
    assert [r['input'] for r in ROWS[cond]] == SRC, f'{cond}: not paired with {SPLIT}.jsonl'
    blank = sum(1 for r in ROWS[cond] if not r['prediction'].strip())
    assert blank == 0, f'{cond}: {blank} blank predictions break the feature matrix pairing'
    print(f'{cond:10s} {len(ROWS[cond])} rows, aligned, 0 blank')

identical = sum(1 for a, b in zip(ROWS[CONTROL], ROWS[ARM]) if a['prediction'] == b['prediction'])
print(f'\n{identical / len(VAL):.1%} of segments are decoded identically by {CONTROL} and {ARM}: '
      f'the two differ only in which exemplars the prompt carried.')

peft       1323 rows, aligned, 0 blank
peft_knn   1323 rows, aligned, 0 blank
peft_afsp  1323 rows, aligned, 0 blank

11.3% of segments are decoded identically by peft_knn and peft_afsp: the two differ only in which exemplars the prompt carried.


In [137]:
MANIFEST = json.loads((OUT / 'peft_afsp_manifest.json').read_text(encoding='utf-8'))
PEFT_CFG = yaml.safe_load(Path('configs/peft_qwen.yaml').read_text(encoding='utf-8'))
GEN, PEFT_GEN = MANIFEST['generator'], PEFT_CFG['generator']

for key in ('model', 'adapter_path', 'temperature', 'top_p', 'seed', 'max_tokens',
            'dtype', 'load_in_4bit'):
    assert GEN[key] == PEFT_GEN[key], f'{key}: {GEN[key]} vs peft {PEFT_GEN[key]}'
assert set(MANIFEST['conditions']) == {CONTROL, ARM}, MANIFEST['conditions']

idx = MANIFEST['index']
print('decoding matches peft on all eight settings')
print(f"adapter {MANIFEST['adapter']['sha256'][:12]}  r={MANIFEST['adapter']['r']}")
print(f"index rebuilt_here={idx['rebuilt_here']}  reproduced={idx.get('reproduced')}")
print(f"device {MANIFEST['versions']['device']}, torch {MANIFEST['versions']['torch']}")
if idx.get('reproduced') is False:
    print('\nWARNING: the index differs from the one afsp_full used. The contrasts below are'
          '\nunaffected -- all three conditions come from one box -- but any comparison to the'
          '\nJuly afsp_full row must say so.')

decoding matches peft on all eight settings
adapter ad97c46af852  r=32
index rebuilt_here=False  reproduced=None
device NVIDIA GeForce RTX 4090, torch 2.12.0+cu130


---
## 2 — The detection floor

In [138]:
DECOMP = json.loads(Path(f'results/heldout_decomp_{SPLIT}.json').read_text(encoding='utf-8'))
PROMPTING = json.loads(
    Path(f'results/heldout_decomp_prompting_{SPLIT}.json').read_text(encoding='utf-8'))

for report in (DECOMP, PROMPTING):
    assert report['reference'] == REFERENCE, report['reference']
    boot = report['bootstrap']
    assert (boot['n_resamples'], boot['seed'], boot['alpha']) == (N_BOOT, SEED, ALPHA), boot
    assert boot['n_segments'] == len(VAL), boot

BASE = PROMPTING['cells'][REFERENCE]['dist_heldout']
CEILING = PROMPTING['cells']['afsp_full']['dist_heldout']
HALF = {c: (v['dist_heldout_delta']['ci_high'] - v['dist_heldout_delta']['ci_low']) / 2
        for c, v in DECOMP['cells'].items() if 'dist_heldout_delta' in v}
FLOOR = max(HALF.values())

for cond, h in HALF.items():
    print(f'{cond:14s} half-width against {REFERENCE} {h:.4f}')
print(f'\nfloor {FLOOR:.4f}: a shift of that size or below is not distinguishable from resampling '
      f'noise at n={len(VAL)}.')
print(f'P1 predicts {ARM} lands between peft {BASE:.4f} and afsp_full {CEILING:.4f}.')

rlsf_w3_0.0    half-width against peft 0.0186
rlsf_w3_2.0    half-width against peft 0.0245
rlsf_w3_6.0    half-width against peft 0.0204

floor 0.0245: a shift of that size or below is not distinguishable from resampling noise at n=1323.
P1 predicts peft_afsp lands between peft 0.1707 and afsp_full 0.2990.


---
## 3 — Adequacy: chrF, BLEU, COMET

In [139]:
!{PY} manage.py eval --conditions {' '.join(CONDS)} --split {SPLIT}

condition  n     BLEU   chrF   marker_rate  ref_marker_rate
-----------------------------------------------------------
peft       1323  16.9   41.58  0.92         0.79           
peft_knn   1323  17.98  42.4   0.78         0.79           
peft_afsp  1323  17.77  42.12  0.79         0.79           


In [140]:
from src.eval.quick import score

SURFACE = {c: score(c, OUT, SPLIT) for c in CONDS}
print(f"{'condition':12s} {'chrF':>8s} {'BLEU':>8s} {'markers/seg':>12s}")
for cond in CONDS:
    s = SURFACE[cond]
    print(f"{cond:12s} {s['chrF']:8.2f} {s['BLEU']:8.2f} {s['marker_rate']:12.2f}")
print(f"gold targets carry {SURFACE[REFERENCE]['ref_marker_rate']:.2f} markers per segment")

condition        chrF     BLEU  markers/seg
peft            41.58    16.90         0.92
peft_knn        42.40    17.98         0.78
peft_afsp       42.12    17.77         0.79
gold targets carry 0.79 markers per segment


In [141]:
if not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install', '-q']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)

COMET_PATH = f'results/comet_{SPLIT}.json'
PRIOR = set(json.loads(Path(COMET_PATH).read_text(encoding='utf-8')))
for cond in (CONTROL, ARM):
    subprocess.run(
        [COMET_PY, 'manage.py', 'comet', '--conditions', cond, '--split', SPLIT,
         '--results_path', COMET_PATH, '--batch_size', '16'],
        check=True,
    )

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 2231.73it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/prnamhr/projects/Style-Aware-MT/.venv-comet/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and ver

peft_knn         COMET 0.7015  (n=1323)
preserved 11 condition(s) not scored here: afsp_full, afsp_margin, commercial_haiku, knn_fewshot, peft, peft_afsp, random_fewshot, rlsf_w3_0.0, rlsf_w3_2.0, rlsf_w3_6.0, zeroshot
Wrote results/comet_val.json


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1752.01it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/prnamhr/projects/Style-Aware-MT/.venv-comet/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and ver

peft_afsp        COMET 0.7033  (n=1323)
preserved 11 condition(s) not scored here: afsp_full, afsp_margin, commercial_haiku, knn_fewshot, peft, peft_knn, random_fewshot, rlsf_w3_0.0, rlsf_w3_2.0, rlsf_w3_6.0, zeroshot
Wrote results/comet_val.json


In [142]:
COMET = json.loads(Path(COMET_PATH).read_text(encoding='utf-8'))

assert PRIOR <= set(COMET), f'lost from {COMET_PATH}: {sorted(PRIOR - set(COMET))}'
ref = COMET[REFERENCE]
for cond in (CONTROL, ARM):
    rec = COMET[cond]
    assert rec['n'] == len(VAL), (cond, rec['n'])
    assert rec['model'] == ref['model'], (cond, rec['model'])
    assert rec['sources'] == ref['sources'], f'{cond}: segments are not paired with {REFERENCE}'

print(f"{ref['model']} scored all three on the same {len(VAL)} segments; "
      f'{len(PRIOR)} conditions already in the file preserved')
for cond in CONDS:
    print(f"{cond:12s} COMET {COMET[cond]['system']:.4f}")

Unbabel/wmt22-comet-da scored all three on the same 1323 segments; 12 conditions already in the file preserved
peft         COMET 0.6986
peft_knn     COMET 0.7015
peft_afsp    COMET 0.7033


In [143]:
# One paired bootstrap per metric, each writing its own table. --adjacent adds the
# peft_afsp - peft_knn pair that P3 is scored on; the other two are against peft.
for metric in METRICS:
    r = subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', metric, '--split', SPLIT,
                        '--adjacent', '--conditions', *CONDS, '--baseline', REFERENCE,
                        '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                        '--out', f'results/bootstrap_{metric}_peft_afsp_{SPLIT}.json'],
                       check=False)
    assert r.returncode == 0, f'{metric} bootstrap exited {r.returncode}'

wrote results/bootstrap_chrf_peft_afsp_val.json

chrf paired bootstrap  (resamples=10000, split=val)
comparison            n     diff    ci95             p       sig
----------------------------------------------------------------
peft_knn - peft       1323  0.964   [0.403, 1.527]   0.001   *  
peft_afsp - peft      1323  0.901   [0.330, 1.473]   0.0014  *  
peft_afsp - peft_knn  1323  -0.063  [-0.575, 0.465]  0.831      

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/bootstrap_bleu_peft_afsp_val.json

bleu paired bootstrap  (resamples=10000, split=val)
comparison            n     diff    ci95             p       sig
----------------------------------------------------------------
peft_knn - peft       1323  0.825   [0.262, 1.384]   0.0048  *  
peft_afsp - peft      1323  0.759   [0.201, 1.308]   0.0062  *  
peft_afsp - peft_knn  1323  -0.066  [-0.603, 0.476]  0.8248     

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/bootstrap_comet_pe

---
## 4 — Register fit

In [144]:
!{PY} manage.py stylometrics --conditions {' '.join(CONDS)} --split {SPLIT} --targets-split train

label         n      lex_density  lex_density_sd  ttr     ttr_sd  root_ttr  root_ttr_sd  sent_len_mean  sent_len_mean_sd  sent_len_var  sent_len_var_sd  marker_rate  marker_rate_sd  stylo_dist
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
target:train  10860  0.4344       0.1101          0.854   0.1085  4.0437    1.0426       24.7388        16.6125           6.6894        58.4011          0.0327       0.0567          0.0       
peft          1323   0.4088       0.1118          0.8515  0.1229  3.9389    1.0278       25.4086        29.1189           3.6097        38.2494          0.0404       0.0621          0.2886    
peft_knn      1323   0.4108       0.1071          0.8366  0.1242  3.8745    0.9858       24.8611        17.0783           3.148         42.3741          0.0347       0.0588          0.3152    
peft_afsp     1323   0.4134       0

In [145]:
!{PY} manage.py stylometrics_ci --split {SPLIT} --conditions {' '.join(CONDS)} \
    --n_resamples {N_STYLO} --alpha {ALPHA} --seed {SEED} \
    --results_path results/stylometrics_ci_peft_afsp_{SPLIT}.json


Register fit of the main conditions  (split=val, n=1323 segments, resamples=2000, seed=42)
stylo_dist = standardized distance to the target-register centroid; lower is better.

rank  condition  stylo_dist  ci95              P(this rank)  modal rank  mean rank
----------------------------------------------------------------------------------
1     peft_afsp  0.2701      [0.2292, 0.3217]  0.796         1 (0.796)   1.20     
2     peft       0.2886      [0.2410, 0.3466]  0.662         2 (0.662)   1.93     
3     peft_knn   0.3152      [0.2730, 0.3666]  0.864         3 (0.864)   2.86     

Signed z per register feature (95% CI; 0 = on target)
condition  lex_density              ttr                      root_ttr                 marker_rate           
------------------------------------------------------------------------------------------------------------
peft_afsp  -0.191 [-0.242, -0.136]  -0.109 [-0.171, -0.051]  -0.148 [-0.198, -0.098]  +0.051 [-0.004, 0.107]
peft       -0.233 [-0.289

In [146]:
new = json.loads(Path(f'results/stylometrics_ci_peft_afsp_{SPLIT}.json').read_text(encoding='utf-8'))
old = json.loads(Path(f'results/stylometrics_ci_{SPLIT}.json').read_text(encoding='utf-8'))
a, b = new['cells'][REFERENCE], old['cells'][REFERENCE]
assert a['stylo_dist'] == b['stylo_dist'] and a['z'] == b['z'], 'peft moved between passes'
print(f"peft reproduces the committed row: stylo_dist {a['stylo_dist']:.4f}")

peft reproduces the committed row: stylo_dist 0.2886


---
## 5 — The declared axis: held-out register distance

In [147]:
!{PY} manage.py heldout_decomp --split {SPLIT} --no-figure \
    --conditions {' '.join(CONDS)} --reference {REFERENCE} \
    --n_resamples {N_BOOT} --alpha {ALPHA} --seed {SEED} \
    --results_path results/heldout_decomp_peft_afsp_{SPLIT}.json


Held-out register distance, decomposed  (split=val, n=1323 segments, resamples=10000, seed=42)
distance over ttr, root_ttr, marker_rate; delta and share are against peft. Lower distance is better.

condition  w3  feature      z        z ci95            d vs peft  z^2      share   dist  
-----------------------------------------------------------------------------------------
peft       0   ttr          -0.0227  [-0.084, +0.038]  +0.0000    0.00052    1.8%  0.1707
peft       0   root_ttr     -0.1005  [-0.154, -0.046]  +0.0000    0.01010   34.7%  0.1707
peft       0   marker_rate  +0.1361  [+0.080, +0.195]  +0.0000    0.01852   63.6%  0.1707
peft_knn   0   ttr          -0.1599  [-0.222, -0.098]  -0.1373*   0.02557   48.1%  0.2306
peft_knn   0   root_ttr     -0.1623  [-0.214, -0.111]  -0.0619*   0.02635   49.5%  0.2306
peft_knn   0   marker_rate  +0.0356  [-0.018, +0.090]  -0.1005*   0.00126    2.4%  0.2306
peft_afsp  0   ttr          -0.1094  [-0.170, -0.049]  -0.0869*   0.01197   32.9%

In [148]:
!{PY} manage.py heldout_decomp --split {SPLIT} --no-figure \
    --conditions {CONTROL} {ARM} --reference {CONTROL} \
    --n_resamples {N_BOOT} --alpha {ALPHA} --seed {SEED} \
    --results_path results/heldout_decomp_peft_afsp_vs_knn_{SPLIT}.json


Held-out register distance, decomposed  (split=val, n=1323 segments, resamples=10000, seed=42)
distance over ttr, root_ttr, marker_rate; delta and share are against peft_knn. Lower distance is better.

condition  w3  feature      z        z ci95            d vs peft_knn  z^2      share   dist  
---------------------------------------------------------------------------------------------
peft_knn   0   ttr          -0.1599  [-0.222, -0.098]  +0.0000        0.02557   48.1%  0.2306
peft_knn   0   root_ttr     -0.1623  [-0.214, -0.111]  +0.0000        0.02635   49.5%  0.2306
peft_knn   0   marker_rate  +0.0356  [-0.018, +0.090]  +0.0000        0.00126    2.4%  0.2306
peft_afsp  0   ttr          -0.1094  [-0.170, -0.049]  +0.0504*       0.01197   32.9%  0.1908
peft_afsp  0   root_ttr     -0.1477  [-0.199, -0.097]  +0.0144        0.02182   60.0%  0.1908
peft_afsp  0   marker_rate  +0.0510  [-0.005, +0.109]  +0.0156        0.00261    7.2%  0.1908

Reward-side features, same decomposition aga

In [149]:
D_REF = json.loads(Path(f'results/heldout_decomp_peft_afsp_{SPLIT}.json').read_text(encoding='utf-8'))
D_CTL = json.loads(
    Path(f'results/heldout_decomp_peft_afsp_vs_knn_{SPLIT}.json').read_text(encoding='utf-8'))
assert D_REF['cells'][REFERENCE]['dist_heldout'] == DECOMP['cells'][REFERENCE]['dist_heldout']
assert D_REF['adjacent_in_omega'] == [], 'these rungs share a judge weight; no omega contrast'
print(f"peft dist_heldout {D_REF['cells'][REFERENCE]['dist_heldout']:.4f}, matches the committed file")

peft dist_heldout 0.1707, matches the committed file


---
## 6 — Read-out

In [150]:
def holm(tests, alpha=ALPHA):
    """Holm-Bonferroni over one claim's family; returns name -> (p, survives)."""
    ordered = sorted(tests.items(), key=lambda kv: kv[1])
    out, blocked = {}, False
    for i, (name, p) in enumerate(ordered):
        ok = p <= alpha / (len(ordered) - i)
        blocked = blocked or not ok
        out[name] = (p, not blocked)
    return out


def line(label, delta, lo, hi, p, sig, places=4):
    mark = '*' if sig else ' '
    return f'  {label:26s} {delta:+.{places}f} [{lo:+.{places}f}, {hi:+.{places}f}]  p={p:.4f} {mark}'


PLACES = {'chrf': 2, 'bleu': 2, 'comet': 4}
BOOT = {m: json.loads(
    Path(f'results/bootstrap_{m}_peft_afsp_{SPLIT}.json').read_text(encoding='utf-8'))
        for m in METRICS}
STYLO = json.loads(Path(f'results/stylometrics_ci_peft_afsp_{SPLIT}.json').read_text(encoding='utf-8'))


def comparison(metric, a, b):
    for rec in BOOT[metric]['comparisons']:
        if (rec['a'], rec['b']) == (a, b):
            return rec
    raise KeyError(f'{metric}: {a} - {b} not in the table')

In [151]:
# P1 -- held-out register distance worsens against peft, by the marker_rate mechanism.
d = D_REF['cells'][ARM]['dist_heldout_delta']
arm = D_REF['cells'][ARM]['dist_heldout']

print('P1  dist_heldout, peft_afsp against peft')
print(f'  peft {BASE:.4f} -> peft_afsp {arm:.4f}   (afsp_full on the frozen base: {CEILING:.4f})')
print(line('peft_afsp - peft', d['delta'], d['ci_low'], d['ci_high'], d['p_value'], d['significant']))
c = D_REF['cells'][CONTROL]['dist_heldout_delta']
print(line('peft_knn - peft', c['delta'], c['ci_low'], c['ci_high'], c['p_value'], c['significant']))

worsens = d['significant'] and d['delta'] > 0
between = BASE < arm < CEILING
print(f'\n  predicted: rises above {BASE:.4f}, below {CEILING:.4f}, interval clear of zero')
print(f"  observed move {d['delta']:+.4f} against a {FLOOR:.4f} floor")
print(f"  P1 {'HOLDS' if (worsens and between) else 'FAILS'}"
      f"{'' if d['significant'] else '  (interval straddles zero -- no movement detected)'}")

# The three held-out features, read beside where afsp_full sits on each: P1 names the
# direction, not just the size, and marker_rate is the one it names.
print(f"\n  {'feature':12s} {'peft':>8s} {'peft_afsp':>10s} {'afsp_full':>10s} {'share':>7s}")
for name in ('ttr', 'root_ttr', 'marker_rate'):
    f = D_REF['cells'][ARM]['features'][name]
    r = D_REF['cells'][REFERENCE]['features'][name]
    t = PROMPTING['cells']['afsp_full']['features'][name]
    print(f"  {name:12s} {r['z']:+8.4f} {f['z']:+10.4f} {t['z']:+10.4f} {f['share']:7.1%}")

P1  dist_heldout, peft_afsp against peft
  peft 0.1707 -> peft_afsp 0.1908   (afsp_full on the frozen base: 0.2990)
  peft_afsp - peft           +0.0202 [-0.0289, +0.0713]  p=0.4344  
  peft_knn - peft            +0.0589 [-0.0004, +0.1170]  p=0.0512  

  predicted: rises above 0.1707, below 0.2990, interval clear of zero
  observed move +0.0202 against a 0.0245 floor
  P1 FAILS  (interval straddles zero -- no movement detected)

  feature          peft  peft_afsp  afsp_full   share
  ttr           -0.0227    -0.1094    -0.1090   32.9%
  root_ttr      -0.1005    -0.1477    -0.1779   60.0%
  marker_rate   +0.1361    +0.0510    +0.2142    7.2%


In [152]:
h = D_CTL['cells'][ARM]['dist_heldout_delta']
tests = {'dist_heldout': h['p_value']}
print(f'P3  {ARM} against {CONTROL}')
print(line('dist_heldout', h['delta'], h['ci_low'], h['ci_high'], h['p_value'], h['significant']))
for m in METRICS:
    rec = comparison(m, ARM, CONTROL)
    tests[m] = rec['p_value']
    print(line(m, rec['diff'], rec['ci_low'], rec['ci_high'], rec['p_value'], rec['significant'],
               PLACES[m]))
for rec in STYLO['paired_all']:
    if {rec['a'], rec['b']} != {ARM, CONTROL}:
        continue
    flip = -1.0 if rec['a'] == CONTROL else 1.0
    lo, hi = sorted((flip * rec['ci_low'], flip * rec['ci_high']))
    tests['stylo_dist'] = rec['p_value']
    print(line('stylo_dist', flip * rec['diff'], lo, hi, rec['p_value'], rec['significant']))
assert len(tests) == 5, f'expected five quantities, got {sorted(tests)}'


P3  peft_afsp against peft_knn
  dist_heldout               -0.0387 [-0.0714, -0.0067]  p=0.0190 *
  chrf                       -0.06 [-0.58, +0.46]  p=0.8310  
  bleu                       -0.07 [-0.60, +0.48]  p=0.8248  
  comet                      +0.0017 [-0.0013, +0.0047]  p=0.2614  
  stylo_dist                 -0.0442 [-0.0771, -0.0127]  p=0.0040 *


In [153]:
print('P4  adequacy against peft')
p4 = {}
for m in METRICS:
    for cond in (CONTROL, ARM):
        rec = comparison(m, cond, REFERENCE)
        p4[f'{m}:{cond}'] = rec
        print(line(f'{cond} - peft, {m}', rec['diff'], rec['ci_low'], rec['ci_high'],
                   rec['p_value'], rec['significant'], PLACES[m]))

declared = {k: r for k, r in p4.items() if not k.startswith('comet')}
rose = [k for k, r in declared.items() if r['significant'] and r['diff'] > 0]
print('\n  predicted: chrF and BLEU hold or fall, no significant rise')
print(f"  P4 {'FAILS -- ' + ', '.join(rose) + ' rose' if rose else 'HOLDS'}")

# Corpus against segment mean, the two aggregates of section 3. They can disagree in sign on
# a small gap; the selection rule ranks on the corpus column.
print(f"\n  {'pair':22s} {'corpus':>9s} {'segment mean':>14s}")
for m in ('chrf', 'bleu'):
    key = {'chrf': 'chrF', 'bleu': 'BLEU'}[m]
    for cond in (CONTROL, ARM):
        rec = comparison(m, cond, REFERENCE)
        corpus = SURFACE[cond][key] - SURFACE[REFERENCE][key]
        print(f"  {cond + ' - peft, ' + m:22s} {corpus:+9.2f} {rec['diff']:+14.2f}")

P4  adequacy against peft
  peft_knn - peft, chrf      +0.96 [+0.40, +1.53]  p=0.0010 *
  peft_afsp - peft, chrf     +0.90 [+0.33, +1.47]  p=0.0014 *
  peft_knn - peft, bleu      +0.82 [+0.26, +1.38]  p=0.0048 *
  peft_afsp - peft, bleu     +0.76 [+0.20, +1.31]  p=0.0062 *
  peft_knn - peft, comet     +0.0029 [-0.0008, +0.0066]  p=0.1208  
  peft_afsp - peft, comet    +0.0047 [+0.0011, +0.0081]  p=0.0104 *

  predicted: chrF and BLEU hold or fall, no significant rise
  P4 FAILS -- chrf:peft_knn, chrf:peft_afsp, bleu:peft_knn, bleu:peft_afsp rose

  pair                      corpus   segment mean
  peft_knn - peft, chrf      +0.82          +0.96
  peft_afsp - peft, chrf     +0.54          +0.90
  peft_knn - peft, bleu      +1.08          +0.82
  peft_afsp - peft, bleu     +0.87          +0.76


---
## 7 — Phi, on peft_afsp only (paid)

In [ ]:
JUDGE_CFG = 'configs/judge_eval.yaml'
JUDGE_RESULTS = f'results/judge_{SPLIT}.json'
JUDGE_USAGE = f'results/judge_{SPLIT}_usage.json'
JUDGE_CI_PATH = f'results/judge_ci_peft_afsp_{SPLIT}.json'

# Phi is bought for the control as well as the arm, so P3 carries a judge term.
PHI_CONDS = [CONTROL, ARM]

PRIOR_JUDGE = json.loads(Path(JUDGE_RESULTS).read_text(encoding='utf-8'))
PRIOR_USAGE = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
PRIOR_SPEND, PRIOR_CALLS = (PRIOR_USAGE['cumulative'][k] for k in ('cost_usd', 'calls'))
assert REFERENCE in PRIOR_JUDGE, f'{REFERENCE} is unjudged, so there is no contrast to buy'
BUY = [c for c in PHI_CONDS if c not in PRIOR_JUDGE]
for cond in PHI_CONDS:
    if cond in PRIOR_JUDGE:
        print(f"{cond} already scored (Phi {PRIOR_JUDGE[cond]['mean']:.4f}); it is not re-bought")

TEMPLATE = Path('prompts/judge_eval.txt').read_text(encoding='utf-8')
FIXED = len(TEMPLATE.format(source='', reference='', prediction=''))


def prompt_chars(cond):
    """Mean judge-prompt length in characters for one condition."""
    path = OUT / f'{cond}_{SPLIT}.jsonl'
    rows = [json.loads(x) for x in path.open(encoding='utf-8') if x.strip()]
    return sum(FIXED + len(r['input']) + len(r['output']) + len(r['prediction'])
               for r in rows) / len(rows)


last = PRIOR_USAGE['session']
per_call = last['cost_usd'] / last['calls']
there = sum(prompt_chars(c) for c in PRIOR_USAGE['conditions']) / len(PRIOR_USAGE['conditions'])
here = sum(prompt_chars(c) for c in PHI_CONDS) / len(PHI_CONDS)
N_CALLS = len(VAL) * len(BUY)
PROJECTED = per_call * N_CALLS

print(f"last session: {PRIOR_USAGE['model']}, {last['calls']} calls, ${last['cost_usd']:.4f}")
print(f"  {last['prompt_tokens'] / last['calls']:.0f} in / "
      f"{last['completion_tokens'] / last['calls']:.0f} out tokens per call, "
      f'${per_call * 1000:.3f} per 1000')
print(f'prompt size {here:.0f} chars/segment here against {there:.0f} there '
      f'({here / there - 1:+.2%})')
print(f"\n{N_CALLS} calls for {', '.join(BUY) or 'nothing left'} project to ${PROJECTED:.2f}")
print(f'cumulative judge spend to date ${PRIOR_SPEND:.2f}')


peft_knn already scored (Phi 2.8020); it is not re-bought
peft_afsp already scored (Phi 2.7952); it is not re-bought
last session: claude-haiku-4-5, 1298 calls, $1.3192
  473 in / 109 out tokens per call, $1.016 per 1000
prompt size 1684 chars/segment here against 1683 there (+0.06%)

0 calls for nothing left project to $0.00
cumulative judge spend to date $6.72


In [156]:
# Left False so a top-to-bottom re-run cannot authorise itself.
SPEND_OK = True
BUDGET_USD = 3.20
N_PILOT = 25

assert PROJECTED <= BUDGET_USD, f'projection ${PROJECTED:.2f} exceeds the ${BUDGET_USD:.2f} cap'
print(f'authorised {SPEND_OK}   cap ${BUDGET_USD:.2f}   pilot {N_PILOT} segments x {len(BUY)} '
      f'(${per_call * N_PILOT * len(BUY):.3f})')


authorised True   cap $3.20   pilot 25 segments x 0 ($0.000)


In [ ]:
if not BUY:
    print('nothing to buy: every condition in PHI_CONDS already carries Phi')
else:
    assert SPEND_OK, 'set SPEND_OK = True in the cell above to authorise the pilot'
    r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', *BUY, '--split', SPLIT,
                        '--config', JUDGE_CFG, '--limit', str(N_PILOT)], check=False)
    assert r.returncode == 0, f'pilot exited {r.returncode}'


nothing to buy: every condition in PHI_CONDS already carries Phi


In [ ]:
_u = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
if _u['conditions'] != sorted(BUY) or _u['limit'] != N_PILOT:
    REVISED = PROJECTED
    print(f"{JUDGE_USAGE} holds {_u['conditions']} at limit {_u['limit']}, not this pilot: "
          f'the pilot segments were already bought, so ${PROJECTED:.2f} stands')
else:
    pilot = _u['session']
    measured = pilot['cost_usd'] / pilot['calls']
    REVISED = measured * N_CALLS
    print(f"pilot {pilot['calls']} calls, {pilot['prompt_tokens'] / pilot['calls']:.0f} in / "
          f"{pilot['completion_tokens'] / pilot['calls']:.0f} out tokens per call, "
          f"${pilot['cost_usd']:.4f}")
    print(f'${measured * 1000:.3f} per 1000 against ${per_call * 1000:.3f} carried over '
          f'({measured / per_call - 1:+.1%})')
    print(f"\nreprices to ${REVISED:.2f}, of which ${pilot['cost_usd']:.4f} is bought")
assert REVISED <= BUDGET_USD, f'repriced ${REVISED:.2f} exceeds the ${BUDGET_USD:.2f} cap'


results/judge_val_usage.json holds ['peft_knn'] at limit None, not this pilot: the pilot segments were already bought, so $0.00 stands


In [159]:
# The client is built on first call, so re-running over a complete cache makes no request.
if not BUY:
    print('nothing to buy; the results file already carries every condition')
else:
    assert SPEND_OK, 'set SPEND_OK = True to authorise the full pass'
    r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', *BUY, '--split', SPLIT,
                        '--config', JUDGE_CFG], check=False)
    assert r.returncode == 0, f'judge exited {r.returncode}'


nothing to buy; the results file already carries every condition


In [160]:
!{PY} manage.py judge_ci --split {SPLIT} --conditions {REFERENCE} {CONTROL} {ARM} \
    --n_resamples {N_BOOT} --alpha {ALPHA} --seed {SEED} \
    --results_path {JUDGE_CI_PATH}



Judge register fidelity Phi by condition  (split=val, n=1323 segments, resamples=10000, seed=42)
judge: claude-haiku-4-5  [tag (none)]
Phi = mean 1-5 rubric rating against the authorized reference; higher is better.

rank  condition  class  n     Phi     ci95              sd     P(this rank)  modal rank  mean rank
--------------------------------------------------------------------------------------------------
1     peft_knn   study  1323  2.8020  [2.7468, 2.8571]  1.028  0.617         1 (0.617)   1.39     
2     peft_afsp  study  1323  2.7952  [2.7400, 2.8503]  1.030  0.602         2 (0.602)   1.64     
3     peft       study  1323  2.7438  [2.6893, 2.7997]  1.033  0.974         3 (0.974)   2.97     

Score distribution over the rubric (share of parsed segments)
condition  coverage  =1     =2     =3     =4     =5   
------------------------------------------------------
peft_knn   1.0000    0.128  0.252  0.317  0.293  0.009
peft_afsp  1.0000    0.129  0.255  0.317  0.288  0.011
peft

In [161]:
# Reads the stored segment scores, so it costs nothing and can be re-run at will.
# --adjacent adds the peft_afsp - peft_knn pair that P3 is scored on, as in section 3.
JBOOT_PATH = f'results/bootstrap_judge_peft_afsp_{SPLIT}.json'
r = subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', 'judge', '--split', SPLIT,
                    '--adjacent', '--conditions', REFERENCE, CONTROL, ARM,
                    '--baseline', REFERENCE,
                    '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                    '--out', JBOOT_PATH],
                   check=False)
assert r.returncode == 0, f'judge bootstrap exited {r.returncode}'
# It exits 0 having compared nothing when a condition is unscored, so check it wrote.
assert Path(JBOOT_PATH).exists(), f'no {JBOOT_PATH}: {BUY} have no judge scores yet'


wrote results/bootstrap_judge_peft_afsp_val.json

judge paired bootstrap  (resamples=10000, split=val)
comparison            n     diff    ci95             p       sig
----------------------------------------------------------------
peft_knn - peft       1323  0.058   [0.009, 0.108]   0.0204  *  
peft_afsp - peft      1323  0.051   [0.003, 0.101]   0.0376  *  
peft_afsp - peft_knn  1323  -0.007  [-0.051, 0.038]  0.7838     

* = 95% CI excludes 0 (difference significant at α=0.05)


In [162]:
JUDGE = json.loads(Path(JUDGE_RESULTS).read_text(encoding='utf-8'))
JCI = json.loads(Path(JUDGE_CI_PATH).read_text(encoding='utf-8'))
JBOOT = json.loads(Path(JBOOT_PATH).read_text(encoding='utf-8'))
OLD_CI = json.loads(Path(f'results/judge_ci_{SPLIT}.json').read_text(encoding='utf-8'))

lost = sorted(set(PRIOR_JUDGE) - set(JUDGE))
assert not lost, f'lost from {JUDGE_RESULTS}: {lost}'
for cond in BUY:
    assert JUDGE[cond]['model'] == JUDGE[REFERENCE]['model'], f'{cond}: two raters, not one'
    assert JUDGE[cond]['sources'] == JUDGE[REFERENCE]['sources'], f'{cond} is not paired'
    assert JUDGE[cond]['coverage'] == 1.0, f"{cond} coverage {JUDGE[cond]['coverage']:.3f}"


def jpair(a, b):
    for rec in JBOOT['comparisons']:
        if (rec['a'], rec['b']) == (a, b):
            return rec
    raise KeyError(f'judge: {a} - {b} not in the table')


# The floor is the half-width this judge produced on the closest analogue at the same n:
# a prompting condition against the adapter.
FLOOR_PHI = next((r['ci_high'] - r['ci_low']) / 2 for r in OLD_CI['paired_adjacent']
                 if {r['a'], r['b']} == {'knn_fewshot', REFERENCE})
phi = jpair(ARM, REFERENCE)

print('P2  Phi, peft_afsp against peft')
for cond in (REFERENCE, CONTROL, ARM):
    lo, hi = JCI['cells'][cond]['phi_ci']
    print(f"  {cond:12s} Phi {JCI['cells'][cond]['phi']:.4f} [{lo:.4f}, {hi:.4f}]")
for a, b in ((ARM, REFERENCE), (CONTROL, REFERENCE)):
    rec = jpair(a, b)
    print(line(f'{a} - {b}', rec['diff'], rec['ci_low'], rec['ci_high'], rec['p_value'],
               rec['significant']))
print(f'\n  predicted: |dPhi| inside the {FLOOR_PHI:.4f} floor, interval covering zero')
inside = abs(phi['diff']) <= FLOOR_PHI and not phi['significant']
print(f"  P2 {'HOLDS' if inside else 'FAILS'}")

# P3's sixth quantity, available now that the control carries Phi. Left as its own dict so
# section 6's five-term family is not mutated from down here.
p3_phi = jpair(ARM, CONTROL)
P3_TESTS = {**tests, 'phi': p3_phi['p_value']}
print(f'\nP3  {ARM} against {CONTROL}, judge term')
print(line('phi', p3_phi['diff'], p3_phi['ci_low'], p3_phi['ci_high'], p3_phi['p_value'],
           p3_phi['significant']))
print(f'  P3 now carries {len(P3_TESTS)} quantities: {", ".join(sorted(P3_TESTS))}')


P2  Phi, peft_afsp against peft
  peft         Phi 2.7438 [2.6893, 2.7997]
  peft_knn     Phi 2.8020 [2.7468, 2.8571]
  peft_afsp    Phi 2.7952 [2.7400, 2.8503]
  peft_afsp - peft           +0.0514 [+0.0030, +0.1013]  p=0.0376 *
  peft_knn - peft            +0.0582 [+0.0091, +0.1081]  p=0.0204 *

  predicted: |dPhi| inside the 0.0587 floor, interval covering zero
  P2 FAILS

P3  peft_afsp against peft_knn, judge term
  phi                        -0.0068 [-0.0514, +0.0378]  p=0.7838  
  P3 now carries 6 quantities: bleu, chrf, comet, dist_heldout, phi, stylo_dist


In [163]:
usage = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
spent = usage['cumulative']['cost_usd'] - PRIOR_SPEND
paid_calls = usage['cumulative']['calls'] - PRIOR_CALLS
assert usage['priced'], 'the judge model has no pricing table; cost_usd is a floor, not a bill'
assert spent <= BUDGET_USD, f'${spent:.2f} spent against a ${BUDGET_USD:.2f} cap'
print(f"{paid_calls} paid calls, ${spent:.2f} on {usage['model']} this session for "
      f"{', '.join(BUY) or 'nothing'}; cumulative judge spend "
      f"${usage['cumulative']['cost_usd']:.2f}")
for p in (JUDGE_RESULTS, JUDGE_USAGE, JUDGE_CI_PATH, JBOOT_PATH,
          *(f'results/judge_{SPLIT}_segments/{c}.jsonl' for c in PHI_CONDS)):
    print(f'  {p}  {Path(p).stat().st_size / 1024:.1f} KiB')


0 paid calls, $0.00 on claude-haiku-4-5 this session for nothing; cumulative judge spend $6.72
  results/judge_val.json  6330.2 KiB
  results/judge_val_usage.json  0.4 KiB
  results/judge_ci_peft_afsp_val.json  3.3 KiB
  results/bootstrap_judge_peft_afsp_val.json  1.1 KiB
  results/judge_val_segments/peft_knn.jsonl  215.0 KiB
  results/judge_val_segments/peft_afsp.jsonl  215.0 KiB


---
## 8 — Phi_B, the cross-family second rater (paid)

In [164]:
JUDGE_CFG_B = 'configs/judge_eval_gpt.yaml'
TAG_B = 'gpt'
JUDGE_RESULTS_B = f'results/judge_{TAG_B}_{SPLIT}.json'
JUDGE_USAGE_B = f'results/judge_{TAG_B}_{SPLIT}_usage.json'
JUDGE_CI_PATH_B = f'results/judge_ci_{TAG_B}_peft_afsp_{SPLIT}.json'
JBOOT_PATH_B = f'results/bootstrap_judge_{TAG_B}_peft_afsp_{SPLIT}.json'
FLOOR_PATH_B = f'results/judge_floor_{TAG_B}_{SPLIT}.json'

PRIOR_JUDGE_B = json.loads(Path(JUDGE_RESULTS_B).read_text(encoding='utf-8'))
PRIOR_USAGE_B = json.loads(Path(JUDGE_USAGE_B).read_text(encoding='utf-8'))
PRIOR_SPEND_B, PRIOR_CALLS_B = (PRIOR_USAGE_B['cumulative'][k] for k in ('cost_usd', 'calls'))
assert REFERENCE in PRIOR_JUDGE_B, f'{REFERENCE} is unjudged by {TAG_B}; no contrast to buy'
BUY_B = [c for c in PHI_CONDS if c not in PRIOR_JUDGE_B]
for cond in PHI_CONDS:
    if cond in PRIOR_JUDGE_B:
        print(f"{cond} already scored by {TAG_B} "
              f"(Phi {PRIOR_JUDGE_B[cond]['mean']:.4f}); it is not re-bought")

# Measured over the whole ledger rather than the last session: the batch rate is stable and
# 13k calls price it better than one pass does.
cum_b = PRIOR_USAGE_B['cumulative']
rate_b = cum_b['cost_usd'] / cum_b['calls']
N_CALLS_B = len(VAL) * len(BUY_B)
PROJECTED_B = rate_b * N_CALLS_B

print(f"\n{PRIOR_USAGE_B['model']} over {PRIOR_USAGE_B['transport']} transport, "
      f"{PRIOR_USAGE_B['batch_discount']:.0%} of list")
print(f"  ${rate_b * 1000:.3f} per 1000, measured over {cum_b['calls']} calls")
print(f"\n{N_CALLS_B} calls for {', '.join(BUY_B) or 'nothing left'} "
      f'project to ${PROJECTED_B:.2f}')
print(f'cumulative {TAG_B} spend to date ${PRIOR_SPEND_B:.2f}')


gpt-5.6-terra over batch transport, 50% of list
  $0.659 per 1000, measured over 13230 calls

2646 calls for peft_knn, peft_afsp project to $1.74
cumulative gpt spend to date $8.72


In [165]:
# Both raters must read the same frozen rubric or Phi_A and Phi_B are not comparable.
import hashlib

for path in (JUDGE_CFG, JUDGE_CFG_B):
    c = yaml.safe_load(Path(path).read_text(encoding='utf-8'))
    assert c['template_file'] == 'prompts/judge_eval.txt', c['template_file']
    print(f"{path:30s} {c['judge']['model']:16s} tag={c.get('tag') or '(none)'}")

frozen = json.loads(Path('prompts/hashes.json').read_text(encoding='utf-8'))['templates']
digest = hashlib.sha256(Path('prompts/judge_eval.txt').read_bytes()).hexdigest()
assert digest == frozen['judge_eval.txt']['sha256'], 'the evaluation rubric has drifted'
print(f'\nrubric verified {digest[:16]}')

configs/judge_eval.yaml        claude-haiku-4-5 tag=(none)
configs/judge_eval_gpt.yaml    gpt-5.6-terra    tag=gpt

rubric verified ffd6dad41acb0512


In [168]:
# Left False so a top-to-bottom re-run cannot authorise itself.
SPEND_OK_B = True
BUDGET_B_USD = 2.20

assert PROJECTED_B <= BUDGET_B_USD, (
    f'projection ${PROJECTED_B:.2f} exceeds the ${BUDGET_B_USD:.2f} cap')
print(f'authorised {SPEND_OK_B}   cap ${BUDGET_B_USD:.2f}   projected ${PROJECTED_B:.2f}')

authorised True   cap $2.20   projected $1.74


In [ ]:
if not BUY_B:
    print(f'nothing to buy: every condition in PHI_CONDS already carries Phi_{TAG_B}')
else:
    assert SPEND_OK_B, 'set SPEND_OK_B = True in the cell above to authorise the batch'
    r = subprocess.run([PY, 'manage.py', 'judge_batch', '--conditions', *BUY_B,
                        '--split', SPLIT, '--config', JUDGE_CFG_B], check=False)
    assert r.returncode == 0, f'judge_batch exited {r.returncode}'


judge gpt-5.6-terra [batch]  tag=gpt  template=prompts/judge_eval.txt [ffd6dad41acb0512]
Judging 1323 segments for peft_knn with gpt-5.6-terra ...
  submitting 1323 requests ...
  submitted batch batch_6a86b3c1fd988190a8633b15b3374655
  [validating] 0/0 completed
  [validating] 0/0 completed
  [in_progress] 0/1323 completed
  [in_progress] 0/1323 completed
  [in_progress] 365/1323 completed
  [in_progress] 741/1323 completed
  [in_progress] 1209/1323 completed
  [finalizing] 1323/1323 completed
  [finalizing] 1323/1323 completed
  [completed] 1323/1323 completed
  batch completed: wrote 1323 segment(s), 27 unscored
  peft_knn         Φ 3.662  (coverage 98%)
Judging 1323 segments for peft_afsp with gpt-5.6-terra ...
  submitting 1323 requests ...
  submitted batch batch_6a86b4db79d08190a652cf38bd7ca60d
  [validating] 0/0 completed
  [in_progress] 0/1323 completed
  [in_progress] 127/1323 completed
  [in_progress] 550/1323 completed
  [in_progress] 1053/1323 completed
  [finalizing] 1323

In [170]:
!{PY} manage.py judge_ci --split {SPLIT} --conditions {REFERENCE} {CONTROL} {ARM} \
    --tag {TAG_B} --n_resamples {N_BOOT} --alpha {ALPHA} --seed {SEED} \
    --results_path {JUDGE_CI_PATH_B}


Judge register fidelity Phi by condition  (split=val, n=1323 segments, resamples=10000, seed=42)
judge: gpt-5.6-terra  [tag gpt]
Phi = mean 1-5 rubric rating against the authorized reference; higher is better.

rank  condition  class  n     Phi     ci95              sd     P(this rank)  modal rank  mean rank
--------------------------------------------------------------------------------------------------
1     peft_knn   study  1296  3.6620  [3.6101, 3.7150]  0.967  0.856         1 (0.856)   1.16     
2     peft_afsp  study  1299  3.6366  [3.5834, 3.6905]  0.986  0.710         2 (0.710)   2.03     
3     peft       study  1306  3.6126  [3.5567, 3.6682]  1.020  0.829         3 (0.829)   2.82     

Score distribution over the rubric (share of parsed segments)
condition  coverage  =1     =2     =3     =4     =5   
------------------------------------------------------
peft_knn   0.9796    0.050  0.074  0.171  0.574  0.131
peft_afsp  0.9819    0.053  0.077  0.184  0.552  0.134
peft      

In [171]:
# Reads the stored segment scores, so both calls are free and can be re-run at will.
r = subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', 'judge', '--split', SPLIT,
                    '--judge_tag', TAG_B, '--adjacent',
                    '--conditions', REFERENCE, CONTROL, ARM, '--baseline', REFERENCE,
                    '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                    '--out', JBOOT_PATH_B], check=False)
assert r.returncode == 0, f'{TAG_B} judge bootstrap exited {r.returncode}'
assert Path(JBOOT_PATH_B).exists(), f'no {JBOOT_PATH_B}: {BUY_B} have no {TAG_B} scores yet'

# knn_fewshot and peft are not rank-adjacent under this rater, so the floor section 7 reads
# off judge_ci has to be measured directly here.
r = subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', 'judge', '--split', SPLIT,
                    '--judge_tag', TAG_B, '--conditions', 'knn_fewshot', REFERENCE,
                    '--baseline', REFERENCE, '--pairs', f'knn_fewshot:{REFERENCE}',
                    '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                    '--out', FLOOR_PATH_B], check=False)
assert r.returncode == 0, f'{TAG_B} floor bootstrap exited {r.returncode}'


wrote results/bootstrap_judge_gpt_peft_afsp_val.json

judge paired bootstrap  (resamples=10000, split=val)
comparison            n     diff    ci95             p       sig
----------------------------------------------------------------
peft_knn - peft       1280  0.048   [0.001, 0.097]   0.0484  *  
peft_afsp - peft      1283  0.023   [-0.024, 0.072]  0.352      
peft_afsp - peft_knn  1273  -0.027  [-0.072, 0.019]  0.2406     

* = 95% CI excludes 0 (difference significant at α=0.05)
wrote results/judge_floor_gpt_val.json

judge paired bootstrap  (resamples=10000, split=val)
comparison          n     diff   ci95            p       sig
------------------------------------------------------------
knn_fewshot - peft  1291  0.067  [0.010, 0.125]  0.0214  *  

* = 95% CI excludes 0 (difference significant at α=0.05)


In [172]:
JUDGE_B = json.loads(Path(JUDGE_RESULTS_B).read_text(encoding='utf-8'))
JCI_B = json.loads(Path(JUDGE_CI_PATH_B).read_text(encoding='utf-8'))
JBOOT_B = json.loads(Path(JBOOT_PATH_B).read_text(encoding='utf-8'))
FLOOR_B_RAW = json.loads(Path(FLOOR_PATH_B).read_text(encoding='utf-8'))

# This rater leaves a verdict unparsed on roughly 1-2% of segments, so coverage is held to a
# floor rather than to 1.0 as under the primary judge.
COVERAGE_MIN_B = 0.97
lost = sorted(set(PRIOR_JUDGE_B) - set(JUDGE_B))
assert not lost, f'lost from {JUDGE_RESULTS_B}: {lost}'
for cond in PHI_CONDS:
    assert JUDGE_B[cond]['model'] == JUDGE_B[REFERENCE]['model'], f'{cond}: two raters, not one'
    assert JUDGE_B[cond]['coverage'] >= COVERAGE_MIN_B, (
        f"{cond} coverage {JUDGE_B[cond]['coverage']:.4f} under {COVERAGE_MIN_B}")


def jpair_b(a, b):
    for rec in JBOOT_B['comparisons']:
        if (rec['a'], rec['b']) == (a, b):
            return rec
    raise KeyError(f'{TAG_B}: {a} - {b} not in the table')


_f = FLOOR_B_RAW['comparisons'][0]
FLOOR_PHI_B = (_f['ci_high'] - _f['ci_low']) / 2
phi_b = jpair_b(ARM, REFERENCE)

print(f'P2  Phi_{TAG_B}, {ARM} against {REFERENCE}')
for cond in (REFERENCE, CONTROL, ARM):
    lo, hi = JCI_B['cells'][cond]['phi_ci']
    print(f"  {cond:12s} Phi {JCI_B['cells'][cond]['phi']:.4f} [{lo:.4f}, {hi:.4f}]"
          f"  coverage {JUDGE_B[cond]['coverage']:.4f}")
for a, b in ((ARM, REFERENCE), (CONTROL, REFERENCE)):
    rec = jpair_b(a, b)
    print(line(f'{a} - {b}', rec['diff'], rec['ci_low'], rec['ci_high'], rec['p_value'],
               rec['significant']))
print(f'\n  predicted: |dPhi| inside the {FLOOR_PHI_B:.4f} floor, interval covering zero')
inside_b = abs(phi_b['diff']) <= FLOOR_PHI_B and not phi_b['significant']
print(f"  P2 under {TAG_B} {'HOLDS' if inside_b else 'FAILS'}")

p3_phi_b = jpair_b(ARM, CONTROL)
print(f'\nP3  {ARM} against {CONTROL}, judge term under {TAG_B}')
print(line('phi', p3_phi_b['diff'], p3_phi_b['ci_low'], p3_phi_b['ci_high'],
           p3_phi_b['p_value'], p3_phi_b['significant']))

# Whether the two raters agree on P2 is the reason for buying the second one.
print(f"\n  {'rater':8s} {'dPhi':>9s} {'floor':>8s} {'p':>8s}  P2")
for name, rec, fl in (('A', phi, FLOOR_PHI), (TAG_B, phi_b, FLOOR_PHI_B)):
    ok = abs(rec['diff']) <= fl and not rec['significant']
    print(f"  {name:8s} {rec['diff']:+9.4f} {fl:8.4f} {rec['p_value']:8.4f}  "
          f"{'HOLDS' if ok else 'FAILS'}")

P2  Phi_gpt, peft_afsp against peft
  peft         Phi 3.6126 [3.5567, 3.6682]  coverage 0.9872
  peft_knn     Phi 3.6620 [3.6101, 3.7150]  coverage 0.9796
  peft_afsp    Phi 3.6366 [3.5834, 3.6905]  coverage 0.9819
  peft_afsp - peft           +0.0234 [-0.0242, +0.0725]  p=0.3520  
  peft_knn - peft            +0.0484 [+0.0008, +0.0969]  p=0.0484 *

  predicted: |dPhi| inside the 0.0577 floor, interval covering zero
  P2 under gpt HOLDS

P3  peft_afsp against peft_knn, judge term under gpt
  phi                        -0.0275 [-0.0723, +0.0189]  p=0.2406  

  rater         dPhi    floor        p  P2
  A          +0.0514   0.0587   0.0376  FAILS
  gpt        +0.0234   0.0577   0.3520  HOLDS


In [173]:
!{PY} manage.py judge_agreement --split {SPLIT} \
    --conditions {REFERENCE} {CONTROL} {ARM} --tag_b {TAG_B} \
    --n_resamples {N_BOOT} --alpha {ALPHA} --seed {SEED} \
    --results_path results/judge_agreement_{TAG_B}_peft_afsp_{SPLIT}.json


Judge-judge agreement  (split=val, resamples=10000, seed=42, 95% percentile CIs)
  judge A: claude-haiku-4-5  [tag (none)]
  judge B: gpt-5.6-terra  [tag gpt]
  same frozen rubric verified by digest: True

Coverage (segments parsed by each rater)
condition         n_total  n_a   n_b   n_both
---------------------------------------------
peft              1323     1323  1306  1306  
peft_knn          1323     1323  1296  1296  
peft_afsp         1323     1323  1299  1299  
commercial_haiku  1323     1323  1308  1308  

Rater agreement -- study_only
condition  n     Phi_A  Phi_B  A-B     ci95              qwk     qwk_ci            rho     exact  adj  
-------------------------------------------------------------------------------------------------------
peft       1306  2.745  3.613  -0.868  [-0.910, -0.824]  +0.518  [+0.483, +0.551]  +0.682  29.9%  81.1%
peft_knn   1296  2.807  3.662  -0.855  [-0.897, -0.813]  +0.511  [+0.476, +0.545]  +0.679  32.4%  81.7%
peft_afsp  1299  2.799  3.637

In [174]:
total = 0.0
for name, path, prior in (('Phi_A', JUDGE_USAGE, PRIOR_SPEND),
                          (f'Phi_{TAG_B}', JUDGE_USAGE_B, PRIOR_SPEND_B)):
    d = json.loads(Path(path).read_text(encoding='utf-8'))
    assert d['priced'], f'{path}: unpriced, so cost_usd is a floor rather than a bill'
    spent_here = d['cumulative']['cost_usd'] - prior
    total += spent_here
    print(f"{name:7s} {d['model']:16s} this session ${spent_here:.4f}  "
          f"| cumulative {d['cumulative']['calls']:6d} calls ${d['cumulative']['cost_usd']:.4f}")
print(f'\nboth raters, this pass ${total:.2f} against '
      f'${BUDGET_USD + BUDGET_B_USD:.2f} authorised across the two caps')

Phi_A   claude-haiku-4-5 this session $0.0000  | cumulative   6590 calls $6.7186
Phi_gpt gpt-5.6-terra    this session $1.7346  | cumulative  15876 calls $10.4566

both raters, this pass $1.73 against $5.40 authorised across the two caps
